In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
"""
EKF + KoopmanMPC — IMPLEMENTACIÓN PLANTA FÍSICA REAL
=====================================================
Esquema de comparación para artículo de journal:
  - Estimador: Extended Kalman Filter (EKF) sobre modelo no lineal nl_ode
  - Controlador: KoopmanMPC (mismo que en koopman_real_plant.py)
  - Planta: ESP32 COM6, sensores HC-SR04 en tanques 1 y 3 (h2 NO medido)

La comparación justa con Koopman Closed-Loop requiere:
  - Mismo KoopmanMPC (A_d, B_d, N, pesos)  ← IDÉNTICO
  - Mismo ESP32, misma planta, mismo escenario de referencias
  - Única diferencia: estimador (EKF aquí vs Luenberger-Koopman allá)

Métricas de interés para el artículo:
  - RMSE seguimiento h2 (estado estacionario, excluyendo transitorios)
  - RMSE estimación h1, h3 (comparados con medición real)
  - Tiempo de cómputo por paso: EKF + MPC vs Luenberger + MPC
  - Número de outliers rechazados (robustez al ruido del ultrasónico)

INSTALACIÓN PREVIA:
  pip install numpy scipy casadi pyserial matplotlib
"""

import numpy as np
from scipy.optimize import least_squares
import scipy.linalg as la
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import serial
import serial.tools.list_ports
import threading
import queue
import sys
import csv
from datetime import datetime

warnings.filterwarnings('ignore')

output_dir = Path('./outputs_real')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS — IDÉNTICOS a koopman_real_plant.py
# ============================================================================
P_NOM = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,
    'eta'     : 8.9e-4,
    'g'       : 9.81,
    'alphao1' : 0.0271, 'Do1': 10e-3,
    'alphao2' : 0.0271, 'A2' : 7.85e-5,
    'alphao3' : 0.0271, 'Do3': 10e-3,
    'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

# ============================================================================
# PARÁMETROS DE EXPERIMENTO — IDÉNTICOS a koopman_real_plant.py
# ============================================================================
Ts         = 2.0
T_sim      = 800
Nsim       = int(T_sim / Ts)
EPS        = 1e-10
EPS_PSI    = 1e-8
SETTLING_S = 60

SERIAL_PORT    = 'COM6'
BAUD_RATE      = 115200
SERIAL_TIMEOUT = 3.0

REF_STEPS = [
    (0,              0.10),
    (int(Nsim*0.33), 0.20),
    (int(Nsim*0.66), 0.15),
]

# ============================================================================
# PARÁMETROS EKF
# ============================================================================
# Covarianza ruido de proceso Q (3x3) — cuánto confías en el modelo
# Diagonal: [varianza_h1, varianza_h2, varianza_h3] en m²
# h2 tiene mayor incertidumbre porque no se mide directamente
EKF_Q = np.diag([
    (0.5e-3)**2,   # h1: ruido de proceso ~0.5 mm
    (2.0e-3)**2,   # h2: mayor incertidumbre (no medido) ~2 mm
    (0.5e-3)**2,   # h3: ruido de proceso ~0.5 mm
])

# Covarianza ruido de medición R (2x2) — ruido del ultrasónico
# HC-SR04 tiene ruido típico de ~1-2 mm
EKF_R = np.diag([
    (1.5e-3)**2,   # h1 medido: ~1.5 mm de ruido
    (1.5e-3)**2,   # h3 medido: ~1.5 mm de ruido
])

# Covarianza inicial P0 — incertidumbre inicial del estado
EKF_P0 = np.diag([
    (5e-3)**2,     # h1: incertidumbre inicial ~5 mm
    (20e-3)**2,    # h2: alta incertidumbre inicial (no medido) ~20 mm
    (5e-3)**2,     # h3: incertidumbre inicial ~5 mm
])

# Filtro anti-outlier del ultrasónico (igual que Koopman)
MAX_DELTA_H = 0.05   # m — salto máximo físicamente posible en 2s

# ============================================================================
# MODELO NO LINEAL — IDÉNTICO a koopman_real_plant.py
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2*p['g']*h1)
    qo2 = p['alphao2'] * p['A2']                   * np.sqrt(2*p['g']*h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2*p['g']*h3)
    dh12 = h1 - h2
    l12  = p['D12']*p['rho']/p['eta'] * np.sqrt(2*p['g']*abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2*l12/p['lambdac12'])
    q12  = a12*p['A12'] * np.sqrt(2*p['g']*abs(dh12)+EPS) * np.sign(dh12+1e-15)
    dh23 = h2 - h3
    l23  = p['D23']*p['rho']/p['eta'] * np.sqrt(2*p['g']*abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2*l23/p['lambdac23'])
    q23  = a23*p['A23'] * np.sqrt(2*p['g']*abs(dh23)+EPS) * np.sign(dh23+1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x + dt/2*k1, u, p)
    k3 = nl_ode(x + dt/2*k2, u, p); k4 = nl_ode(x + dt*k3, u, p)
    return np.clip(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0, 0.27)  # antes 0.25

def psi(x):
    """8 observables de Koopman — IDÉNTICO a koopman_real_plant.py"""
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1*h2, h2*h3, h1*h3, h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, p=P_NOM):
    """
    Punto de operación estacionario para h2=h2_ref.
    Permite u3>0 (regularizado para preferir u3≈0 cuando sea posible),
    ya que para h2=0.20m puede no existir SS factible con u3=0 dentro
    del rango físico [0,0.25]m.
    """
    w_reg = 0.05  # penaliza u3 para preferir soluciones con u3≈0 si existen
    def eqs(v):
        h1, h3, u1, u3 = v
        f = nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                   [np.clip(u1, 0, 1), np.clip(u3, 0, 1)], p)
        return np.concatenate([f, [w_reg * u3]])

    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5, 0.0),
        (h2_ref*1.6, h2_ref*0.7, 0.7, 0.0),
        (h2_ref*2.0, h2_ref*0.8, 0.9, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.7, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.75, 0.0),
        (h2_ref*1.8, h2_ref*0.85, 0.85, 0.0),
        (h2_ref*1.3, h2_ref*0.9, 1.0, 0.1),
        (h2_ref*1.2, h2_ref*1.0, 1.0, 0.3),
        (h2_ref*1.1, h2_ref*1.1, 1.0, 0.5),
        (h2_ref*1.0, h2_ref*1.2, 1.0, 0.8),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0, 0.0],
                                      [0.27, 0.27, 1.0, 1.0]),
                              max_nfev=5000)
            phys_res = np.linalg.norm(s.fun[:3])  # solo las 3 ecuaciones físicas
            if s.success and phys_res < best_res:
                best_res = phys_res; best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return (np.array([best[0], h2_ref, best[1]]),
                np.array([np.clip(best[2], 0, 1), np.clip(best[3], 0, 1)]))
    return None, None

# ============================================================================
# FILTRO DE KALMAN EXTENDIDO (EKF)
# ============================================================================
class EKF:
    """
    Extended Kalman Filter para el sistema de tres tanques.

    Estado: x = [h1, h2, h3]  (metros)
    Medición: y = [h1, h3]     (metros, solo tanques 1 y 3)
    h2 se estima sin medirlo directamente.

    Ciclo por paso de muestreo:
      1. predict(u)  — integra RK4 + propaga covarianza con Jacobiano numérico
      2. update(y)   — corrección bayesiana con medición [h1, h3]
    """

    # Matriz de observación: mide h1 (fila 0) y h3 (fila 2)
    C = np.array([[1.0, 0.0, 0.0],
                  [0.0, 0.0, 1.0]])

    def __init__(self, x0, P0=None, Q=None, R=None, p=P_NOM):
        self.x = np.array(x0, dtype=float)
        self.P = P0.copy() if P0 is not None else EKF_P0.copy()
        self.Q = Q.copy()  if Q  is not None else EKF_Q.copy()
        self.R = R.copy()  if R  is not None else EKF_R.copy()
        self.p = p

    def _jacobian(self, x, u, eps=1e-6):
        """
        Jacobiano numérico Fk = ∂f/∂x evaluado en (x, u).
        Se discretiza como Fk ≈ I + Ts * J_continuo.
        """
        n = len(x)
        J = np.zeros((n, n))
        f0 = nl_ode(x, u, self.p)
        for i in range(n):
            xp = x.copy(); xp[i] += eps
            J[:, i] = (nl_ode(xp, u, self.p) - f0) / eps
        # Jacobiano discreto de primer orden
        return np.eye(n) + Ts * J

    def predict(self, u):
        """
        Etapa de predicción:
          x̂⁻ = RK4(x̂, u)
          P⁻  = Fk @ P @ Fk.T + Q
        """
        Fk      = self._jacobian(self.x, u)
        self.x  = rk4(self.x, u, self.p, Ts)
        self.x  = np.clip(self.x, 0.0, 0.75)
        self.P  = Fk @ self.P @ Fk.T + self.Q

    def update(self, y_meas):
        """
        Etapa de corrección con y = [h1_medido, h3_medido]:
          S  = C @ P⁻ @ C.T + R
          K  = P⁻ @ C.T @ S⁻¹
          x̂  = x̂⁻ + K @ (y - C @ x̂⁻)
          P  = (I - K @ C) @ P⁻
        Devuelve la estimación corregida.
        """
        y    = np.array(y_meas, dtype=float)
        S    = self.C @ self.P @ self.C.T + self.R
        K    = self.P @ self.C.T @ np.linalg.inv(S)
        innov = y - self.C @ self.x
        self.x = self.x + K @ innov
        self.x = np.clip(self.x, 0.0, 0.75)
        self.P = (np.eye(3) - K @ self.C) @ self.P
        # Simetrizar para evitar deriva numérica
        self.P = 0.5 * (self.P + self.P.T)
        return self.x.copy()

    @property
    def h2_estimate(self):
        return float(self.x[1])

    @property
    def std_h2(self):
        """Desviación estándar de la estimación de h2 (raíz de P[1,1])."""
        return float(np.sqrt(max(self.P[1, 1], 0.0)))

# ============================================================================
# COMUNICACIÓN SERIAL — IDÉNTICA a koopman_real_plant.py + filtro outliers
# ============================================================================
class ESP32Interface:
    def __init__(self, port=SERIAL_PORT, baud=BAUD_RATE):
        self.port = port; self.baud = baud; self.ser = None
        self.rx_queue   = queue.Queue(maxsize=20)
        self._stop_event = threading.Event()
        self._last_meas  = None
        self._lock       = threading.Lock()
        self.outlier_count = 0

    def connect(self):
        print(f"\nConectando a {self.port} @ {self.baud} baud...")
        try:
            self.ser = serial.Serial(self.port, self.baud,
                                     timeout=SERIAL_TIMEOUT, write_timeout=1.0)
            time.sleep(2.0)
            self.ser.reset_input_buffer()
            print(f"  ✓ Conectado a {self.port}")
        except serial.SerialException as e:
            print(f"  ✗ Error: {e}")
            for p in serial.tools.list_ports.comports():
                print(f"    {p.device} — {p.description}")
            raise
        self._reader_thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._reader_thread.start()

    def _reader_loop(self):
        while not self._stop_event.is_set():
            try:
                if self.ser.in_waiting:
                    line = self.ser.readline().decode('utf-8', errors='ignore').strip()
                    if line:
                        self.rx_queue.put_nowait(line)
            except Exception: pass
            time.sleep(0.005)

    def get_measurement(self, timeout=3.0):
        """
        Lee (h1, h2_real, h3) en metros con filtro anti-outlier y limpieza de buffer.
        h2_real es la verdad terreno de h2 (medida en la planta física),
        NO se usa como entrada del EKF (que solo mide h1 y h3) — solo sirve
        para validar la estimación del observador contra el valor real.
        Rechaza saltos > MAX_DELTA_H respecto al ultimo valor valido.
        Ignora lineas corruptas (bytes nulos, caracteres no numericos).
        """
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                line = self.rx_queue.get(timeout=0.1)

                # Limpiar bytes nulos que aparecen cuando el buffer serial
                # del ESP32 no se limpio correctamente entre resets
                line = line.replace('\x00', '').replace('\0', '').strip()
                # Eliminar cualquier caracter no imprimible que quede
                line = ''.join(c for c in line if c.isprintable())
                if not line:
                    continue

                parts = line.split(',')
                if len(parts) != 3:
                    continue

                # Convertir con try/except — si falla se descarta la linea
                try:
                    h1      = max(float(parts[0].strip()), 0.0)
                    h2_meas = max(float(parts[1].strip()), 0.0)
                    h3      = max(float(parts[2].strip()), 0.0)
                except (ValueError, OverflowError):
                    continue   # linea corrupta -> esperar la siguiente

                # Filtro anti-outlier
                with self._lock:
                    if self._last_meas is not None:
                        h1p, h2p, h3p = self._last_meas
                        if abs(h1 - h1p) > MAX_DELTA_H:
                            print(f"  Outlier h1: {h1*100:.1f}cm -> usando {h1p*100:.1f}cm")
                            h1 = h1p; self.outlier_count += 1
                        if abs(h2_meas - h2p) > MAX_DELTA_H:
                            print(f"  Outlier h2: {h2_meas*100:.1f}cm -> usando {h2p*100:.1f}cm")
                            h2_meas = h2p; self.outlier_count += 1
                        if abs(h3 - h3p) > MAX_DELTA_H:
                            print(f"  Outlier h3: {h3*100:.1f}cm -> usando {h3p*100:.1f}cm")
                            h3 = h3p; self.outlier_count += 1
                    self._last_meas = (h1, h2_meas, h3)
                return h1, h2_meas, h3

            except queue.Empty:
                pass
        with self._lock:
            if self._last_meas is not None:
                print("  ⚠ Timeout serial — usando última medición")
                return self._last_meas
        raise TimeoutError("No se recibieron datos del ESP32")

    def send_control(self, u1, u2):
        u1 = float(np.clip(u1, 0.0, 1.0))
        u2 = float(np.clip(u2, 0.0, 1.0))
        try:
            self.ser.write(f"{u1:.4f},{u2:.4f}\n".encode('utf-8'))
        except Exception as e:
            print(f"  ✗ Error al enviar: {e}")

    def stop_pumps(self):
        self.send_control(0.0, 0.0); print("  Bombas apagadas.")

    def close(self):
        self._stop_event.set(); self.stop_pumps(); time.sleep(0.5)
        if self.ser and self.ser.is_open: self.ser.close()
        print("  Puerto serial cerrado.")

# ============================================================================
# ENTRENAMIENTO EDMD — IDÉNTICO a koopman_real_plant.py
# ============================================================================
def train_edmd(verbose=True):
    if verbose: print("\nEntrenando EDMD (parámetros nominales)...")
    rng = np.random.default_rng(42)
    z_k, u_k, z_k1 = [], [], []

    for h2r in [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r*pert, 0.05, 0.70)
            for _ in range(170):
                err = x[1]-h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35*err,-0.35,0.35)+rng.uniform(-0.15,0.15)
                u[1] += rng.uniform(-0.05,0.05); u = np.clip(u,0,1)
                z=psi(x); xn=rk4(x,u); zn=psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x=xn

    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.60, 0.70, 0.75, 1.25, 1.35]:
            x = x_ss.copy(); x[1] = np.clip(h2r*pert, 0.05, 0.70)
            for _ in range(100):
                err = x[1]-h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6*err,-0.6,0.6)+rng.uniform(-0.1,0.1)
                u = np.clip(u,0,1)
                z=psi(x); xn=rk4(x,u); zn=psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x=xn

    for (h2f, h2t) in [(0.10,0.20),(0.20,0.15),(0.15,0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None: continue
        x = xf.copy()
        for _ in range(200):
            err = x[1]-h2t; u = uf.copy()
            u[0] += np.clip(-0.5*err,-0.5,0.5)+rng.uniform(-0.08,0.08)
            u = np.clip(u,0,1)
            z=psi(x); xn=rk4(x,u); zn=psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x=xn

    Z=np.array(z_k); U=np.array(u_k); Zn=np.array(z_k1)
    nz, nu = Z.shape[1], U.shape[1]
    if verbose: print(f"  {len(Z)} muestras de entrenamiento")

    Phi=np.hstack([Z,U]); lam=1e-4
    Theta=np.linalg.solve(Phi.T@Phi+lam*np.eye(nz+nu), Phi.T@Zn)
    A_raw=Theta[:nz,:].T; B_d=Theta[nz:,:].T

    vals, vecs = np.linalg.eig(A_raw); delta=1e-4
    vals_s = np.where(np.abs(vals)>=1.0, vals/np.abs(vals)*(1.0-delta), vals)
    A_d = np.real(vecs@np.diag(vals_s)@np.linalg.inv(vecs))

    rmse = np.sqrt(np.mean((Z@A_d.T+U@B_d.T-Zn)**2))
    eigs = np.abs(np.linalg.eigvals(A_d))
    if verbose:
        print(f"  RMSE one-step: {rmse:.2e}")
        print(f"  |λ| ∈ [{eigs.min():.5f}, {eigs.max():.5f}] ✓")
    return A_d, B_d, nz, nu

# ============================================================================
# KoopmanMPC — IDÉNTICO a koopman_real_plant.py (mismos pesos, mismo N)
# ============================================================================
def build_mpc(A_d, B_d, nz, nu):
    print("\nConstruyendo KoopmanMPC...")
    N_mpc=20; q_mpc=500
    R1=np.diag([0.1,1.0]); R2=np.diag([0.1,0.1])
    umin=np.array([0.0,0.0]); umax=np.array([1.0,1.0])

    Z_sym=ca.SX.sym('Z',nz,N_mpc+1); U_sym=ca.SX.sym('U',nu,N_mpc)
    z0_sym=ca.SX.sym('z0',nz); yr_sym=ca.SX.sym('yr'); up_sym=ca.SX.sym('up',nu)
    A_cas=ca.DM(A_d); B_cas=ca.DM(B_d)
    J=0; g=[]; lbg=[]; ubg=[]

    g+=[Z_sym[:,0]-z0_sym]; lbg+=[0.0]*nz; ubg+=[0.0]*nz
    for k in range(N_mpc):
        g+=[Z_sym[:,k+1]-A_cas@Z_sym[:,k]-B_cas@U_sym[:,k]]
        lbg+=[0.0]*nz; ubg+=[0.0]*nz
        h2_k=Z_sym[1,k+1]**2; ey=h2_k-yr_sym
        du=U_sym[:,0]-up_sym if k==0 else U_sym[:,k]-U_sym[:,k-1]
        wk=10 if k==N_mpc-1 else 1
        J+=(wk*q_mpc*ey**2
            +ca.mtimes(U_sym[:,k].T,ca.DM(R1)@U_sym[:,k])
            +ca.mtimes(du.T,ca.DM(R2)@du))

    w_sym=ca.vertcat(ca.reshape(Z_sym,nz*(N_mpc+1),1),ca.reshape(U_sym,nu*N_mpc,1))
    par_sym=ca.vertcat(z0_sym,yr_sym,up_sym)
    lbw=([-ca.inf]*(nz*(N_mpc+1))+list(np.tile(umin,N_mpc)))
    ubw=([ ca.inf]*(nz*(N_mpc+1))+list(np.tile(umax,N_mpc)))

    nlp={'x':w_sym,'f':J,'g':ca.vertcat(*g),'p':par_sym}
    opts={'ipopt.print_level':0,'ipopt.max_iter':500,
          'ipopt.tol':1e-5,'ipopt.acceptable_tol':1e-4,'print_time':0}
    solver=ca.nlpsol('solver_kmpc','ipopt',nlp,opts)
    print("  KoopmanMPC ✓")
    return solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz, nu

# ============================================================================
# LOOP DE CONTROL EN TIEMPO REAL — EKF + KoopmanMPC
# ============================================================================
def run_ekf_experiment(esp, A_d, B_d, solver, N_mpc,
                       umin, umax, lbw, ubw, lbg, ubg, nz, nu, SS_NOM):
    """
    Bucle de control EKF + KoopmanMPC.

    Diferencia clave respecto a koopman_real_plant.py:
      - El estimador es un EKF sobre nl_ode (espacio físico 3D)
      - El estado para el MPC se construye como psi(x_hat_ekf)
      - Todo lo demás (MPC, serial, warm-start) es idéntico
    """
    yref_v = np.zeros(Nsim)
    for (k_start, ref_val) in REF_STEPS:
        yref_v[k_start:] = ref_val
    ref_change_times_s = [k*Ts for k, _ in REF_STEPS[1:]]

    # ── Inicialización desde la primera medición ────────────────────
    print("\nEsperando primera medición del ESP32...")
    h1_meas, h2_real, h3_meas = esp.get_measurement()
    print(f"  Primera medición: h1={h1_meas*100:.2f}cm | h2_real={h2_real*100:.2f}cm | h3={h3_meas*100:.2f}cm")

    # Estado inicial EKF: h2 inicializado en la referencia inicial
    x_ekf_init = np.array([h1_meas, yref_v[0], h3_meas])
    ekf = EKF(x0=x_ekf_init, P0=EKF_P0, Q=EKF_Q, R=EKF_R)

    uprev   = np.array([0.0, 0.0])
    w0_mpc  = None
    yr_prev = yref_v[0]

    # ── Historial ───────────────────────────────────────────────────
    T_hist = np.arange(Nsim+1)*Ts
    H_meas  = np.zeros((2, Nsim+1))  # h1, h3 medidos
    H2_REAL = np.zeros(Nsim+1)       # h2 verdad terreno (NO usado por el EKF)
    X_hat   = np.zeros((3, Nsim+1))  # estimación EKF [h1, h2, h3]
    P_h2    = np.zeros(Nsim+1)       # varianza EKF de h2 (incertidumbre)
    U_hist  = np.zeros((2, Nsim))
    step_ms_ekf = []   # tiempo EKF por paso
    step_ms_mpc = []   # tiempo MPC por paso
    step_ms_tot = []   # tiempo total por paso

    H_meas[0,0]=h1_meas; H_meas[1,0]=h3_meas
    H2_REAL[0]=h2_real
    X_hat[:,0]=x_ekf_init; P_h2[0]=ekf.std_h2**2

    print(f"\n{'='*70}")
    print(f"  EKF + KoopmanMPC — EXPERIMENTO INICIADO {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*70}")
    print(f"  {'t[s]':>6} | {'h1[cm]':>7} | {'ĥ2_EKF':>8} | "
          f"{'h2r[cm]':>7} | {'eobs[cm]':>8} | {'σh2[mm]':>8} | {'h3[cm]':>7} | {'ref[cm]':>7} | "
          f"{'u1':>5} | {'u3':>5} | {'ms':>5}")
    print(f"  {'-'*80}")

    t_exp_start = time.time()

    for k in range(Nsim):
        t_step_start = time.time()
        yr = yref_v[k]

        # ── Warm-start al cambio de referencia ──────────────────────
        if yr != yr_prev and yr in SS_NOM:
            z_ss_new=psi(SS_NOM[yr][0]); u_ss_new=SS_NOM[yr][1]
            Z0=np.tile(z_ss_new,(N_mpc+1,1)); U0=np.tile(u_ss_new,(N_mpc,1))
            w0_mpc=np.concatenate([Z0.flatten(),U0.flatten()])
            print(f"\n  ▶ Ref: {yr_prev*100:.0f}→{yr*100:.0f} cm")
        yr_prev = yr

        # ── 1. EKF: PREDICCIÓN con control anterior ──────────────────
        t_ekf_start = time.time()
        ekf.predict(uprev)

        # ── 2. Leer medición del ESP32 (incluye h2 verdad terreno) ────
        h1_meas, h2_real, h3_meas = esp.get_measurement(timeout=Ts*2)
        H_meas[0,k+1]=h1_meas; H_meas[1,k+1]=h3_meas
        H2_REAL[k+1]=h2_real

        # ── 3. EKF: CORRECCIÓN con medición real ─────────────────────
        x_hat = ekf.update(np.array([h1_meas, h3_meas]))
        X_hat[:,k+1] = x_hat
        P_h2[k+1]    = ekf.std_h2**2
        t_ekf_ms = (time.time() - t_ekf_start)*1000

        # ── 4. Construir estado elevado para el MPC ──────────────────
        # El KoopmanMPC espera z = psi(x_hat_ekf)
        z_for_mpc = psi(x_hat)

        # ── 5. KoopmanMPC ────────────────────────────────────────────
        t_mpc_start = time.time()
        if w0_mpc is None:
            w0_mpc=np.zeros(nz*(N_mpc+1)+nu*N_mpc); w0_mpc[:nz]=z_for_mpc
        par_val=np.concatenate([z_for_mpc,[yr],uprev])
        try:
            sol=solver(x0=w0_mpc,lbx=lbw,ubx=ubw,lbg=lbg,ubg=ubg,p=par_val)
            w_opt=np.array(sol['x']).flatten()
            Z_opt=w_opt[:nz*(N_mpc+1)].reshape(N_mpc+1,nz)
            U_opt=w_opt[nz*(N_mpc+1):].reshape(N_mpc,nu)
            u_k=np.clip(U_opt[0],umin,umax)
            Z_sh=np.vstack([Z_opt[1:],Z_opt[-1:]])
            U_sh=np.vstack([U_opt[1:],U_opt[-1:]])
            w0_mpc=np.concatenate([Z_sh.flatten(),U_sh.flatten()])
        except Exception as e:
            print(f"  ⚠ MPC falló k={k}: {e}")
            u_k=uprev.copy(); w0_mpc=None
        t_mpc_ms = (time.time()-t_mpc_start)*1000

        elapsed_ms = (time.time()-t_step_start)*1000
        step_ms_ekf.append(t_ekf_ms)
        step_ms_mpc.append(t_mpc_ms)
        step_ms_tot.append(elapsed_ms)
        U_hist[:,k]=u_k; uprev=u_k.copy()

        # ── 6. Enviar control al ESP32 ───────────────────────────────
        esp.send_control(u_k[0], u_k[1])

        # ── Log en consola ───────────────────────────────────────────
        if k % 5 == 0:
            e_obs_cm = (x_hat[1] - h2_real) * 100
            print(f"  {k*Ts:>6.0f} | {h1_meas*100:>7.2f} | {x_hat[1]*100:>8.2f} | "
                  f"{h2_real*100:>7.2f} | {e_obs_cm:>+8.3f} | {ekf.std_h2*1000:>8.2f} | "
                  f"{h3_meas*100:>7.2f} | {yr*100:>7.2f} | "
                  f"{u_k[0]:>5.3f} | {u_k[1]:>5.3f} | {elapsed_ms:>5.0f}")

        # ── Sincronizar al ritmo del ESP32 ───────────────────────────
        dt_used = time.time()-t_step_start
        if dt_used < Ts*0.9:
            time.sleep(Ts - dt_used)

    print(f"\n  Experimento finalizado — {(time.time()-t_exp_start)/60:.1f} min")
    print(f"  Outliers rechazados: {esp.outlier_count}")

    return {
        'T_hist'  : T_hist,
        'H_meas'  : H_meas,
        'H2_REAL' : H2_REAL,
        'X_hat'   : X_hat,
        'P_h2'    : P_h2,
        'U_hist'  : U_hist,
        'yref_v'  : yref_v,
        'step_ms_ekf': step_ms_ekf,
        'step_ms_mpc': step_ms_mpc,
        'step_ms_tot': step_ms_tot,
        'ref_change_times_s': ref_change_times_s,
        'outliers': esp.outlier_count,
    }

# ============================================================================
# MÉTRICAS Y GRÁFICAS
# ============================================================================
def compute_metrics(res):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']

    idx_400 = T > 400
    settling_mask = np.ones(len(T), dtype=bool)
    for tc in rct:
        settling_mask &= ~((T>=tc)&(T<tc+SETTLING_S))
    mask_ss = idx_400 & settling_mask

    # Error de seguimiento sobre h2 estimado (lo que controla el MPC)
    e_ctrl = (res['X_hat'][1,:] - ref) * 100

    # Error de estimación de h1 y h3 (comparamos EKF vs medición real)
    e_h1_est = (res['X_hat'][0,:] - res['H_meas'][0,:]) * 100
    e_h3_est = (res['X_hat'][2,:] - res['H_meas'][1,:]) * 100

    # Error de estimación de h2 (EKF vs verdad terreno — NO usado por el EKF)
    e_h2_obs = (res['X_hat'][1,:] - res['H2_REAL']) * 100

    rmse_ctrl   = float(np.sqrt(np.mean(e_ctrl[mask_ss]**2)))
    rmse_h1     = float(np.sqrt(np.mean(e_h1_est[mask_ss]**2)))
    rmse_h3     = float(np.sqrt(np.mean(e_h3_est[mask_ss]**2)))
    rmse_h2_obs = float(np.sqrt(np.mean(e_h2_obs[mask_ss]**2)))
    bias_h2_obs = float(np.mean(e_h2_obs[mask_ss]))
    mae_h2_obs  = float(np.mean(np.abs(e_h2_obs[mask_ss])))
    return (rmse_ctrl, rmse_h1, rmse_h3, e_ctrl, e_h1_est, e_h3_est, mask_ss,
            e_h2_obs, rmse_h2_obs, bias_h2_obs, mae_h2_obs)

def plot_results(res, save_dir=output_dir):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']

    (rmse_ctrl, rmse_h1, rmse_h3, e_ctrl, e_h1_est, e_h3_est, _,
     e_h2_obs, rmse_h2_obs, bias_h2_obs, mae_h2_obs) = compute_metrics(res)
    avg_ekf = np.mean(res['step_ms_ekf'])
    avg_mpc = np.mean(res['step_ms_mpc'])
    avg_tot = np.mean(res['step_ms_tot'])
    p99_tot = np.percentile(res['step_ms_tot'], 99)

    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(
        f'EKF + KoopmanMPC — Real Plant | {datetime.now().strftime("%Y-%m-%d %H:%M")}\n'
        f'RMSE ctrl h₂(SS)={rmse_ctrl:.3f}cm | RMSE obs h₂(SS)={rmse_h2_obs:.3f}cm | '
        f'EKF:{avg_ekf:.1f}ms + MPC:{avg_mpc:.0f}ms = {avg_tot:.0f}ms/step (p99={p99_tot:.0f}ms)',
        fontsize=10, fontweight='bold')

    # h1: measured vs EKF
    ax = axes[0,0]
    ax.plot(T, res['H_meas'][0]*100, 'b', lw=1.5, label='$h_1$ measured')
    ax.plot(T, res['X_hat'][0]*100,  'r--', lw=1.5, label='$h_1$ EKF')
    ax.fill_between(T,
                    (res['X_hat'][0]-2*np.sqrt(res['P_h2']))*100,
                    (res['X_hat'][0]+2*np.sqrt(res['P_h2']))*100,
                    alpha=0.0)   # h1 uncertainty band not plotted (P[0,0] not stored)
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title(f'Tank 1 (measured) — RMSE est={rmse_h1:.3f}cm')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # h2: EKF + uncertainty band + reference + ground truth
    ax = axes[0,1]
    h2_hat = res['X_hat'][1,:]*100
    h2_gt  = res['H2_REAL']*100
    std_h2  = np.sqrt(res['P_h2'])*100
    ax.plot(T, h2_hat, 'r--', lw=2, label='$h_2$ EKF (not used by the EKF)')
    ax.fill_between(T, h2_hat-2*std_h2, h2_hat+2*std_h2,
                    color='red', alpha=0.15, label='±2σ EKF')
    ax.plot(T, h2_gt, 'g-', lw=1.2, alpha=0.85, label='$h_2$ ground truth')
    ax.stairs(ref[:-1]*100, T, color='k', linestyle='--', lw=1.5, label='Reference')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title(f'Tank 2 (ground truth vs EKF) — RMSE obs={rmse_h2_obs:.3f}cm')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # h3: measured vs EKF
    ax = axes[1,0]
    ax.plot(T, res['H_meas'][1]*100, 'b', lw=1.5, label='$h_3$ measured')
    ax.plot(T, res['X_hat'][2]*100,  'r--', lw=1.5, label='$h_3$ EKF')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title(f'Tank 3 (measured) — RMSE est={rmse_h3:.3f}cm')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Tracking error
    ax = axes[1,1]
    ax.plot(T, e_ctrl, 'r', lw=1.5, label=f'RMSE_SS={rmse_ctrl:.3f}cm')
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='excluded transient' if i==0 else None)
    ax.set_ylabel('Ctrl error $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tracking error'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Control signals
    ax = axes[2,0]
    ax.stairs(res['U_hist'][0], T, color='b', lw=1.5, label='$u_1$')
    ax.stairs(res['U_hist'][1], T, color='r', lw=1.5, label='$u_3$')
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]')
    ax.set_xlabel('t [s]'); ax.set_title('KoopmanMPC inputs')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Computation time breakdown
    ax = axes[2,1]
    steps = np.arange(len(res['step_ms_tot']))
    ax.stackplot(steps,
                 res['step_ms_ekf'],
                 res['step_ms_mpc'],
                 labels=[f'EKF ({avg_ekf:.1f}ms)', f'MPC ({avg_mpc:.0f}ms)'],
                 colors=['#2ecc71','#3498db'], alpha=0.8)
    ax.axhline(Ts*1000, color='red', lw=1.5, ls='--', label=f'Ts={Ts*1000:.0f}ms')
    ax.set_ylabel('Time [ms]'); ax.set_xlabel('Step k')
    ax.set_title('Computation time: EKF + MPC (stacked)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = save_dir / f'ekf_koopman_{datetime.now().strftime("%Y%m%d_%H%M%S")}.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Figura guardada: {fname}")

    # CSV con columna extra de incertidumbre EKF en h2
    csv_path = save_dir / f'ekf_datos_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['t_s','h1_meas_m','h1_ekf_m','h2_ekf_m','sigma_h2_m',
                    'h2_real_m','error_obs_cm',
                    'h3_meas_m','h3_ekf_m','ref_m','u1','u3'])
        for k in range(Nsim):
            e_obs_k = (res['X_hat'][1,k] - res['H2_REAL'][k]) * 100
            w.writerow([T[k],
                        res['H_meas'][0,k], res['X_hat'][0,k],
                        res['X_hat'][1,k],  np.sqrt(res['P_h2'][k]),
                        res['H2_REAL'][k],  e_obs_k,
                        res['H_meas'][1,k], res['X_hat'][2,k],
                        res['yref_v'][k],
                        res['U_hist'][0,k], res['U_hist'][1,k]])
    print(f"  Datos guardados:  {csv_path}")

def print_summary(res):
    (rmse_ctrl, rmse_h1, rmse_h3, _, _, _, _,
     _, rmse_h2_obs, bias_h2_obs, mae_h2_obs) = compute_metrics(res)
    avg_ekf = np.mean(res['step_ms_ekf'])
    avg_mpc = np.mean(res['step_ms_mpc'])
    avg_tot = np.mean(res['step_ms_tot'])
    p99_tot = np.percentile(res['step_ms_tot'], 99)

    print(f"\n  {'='*55}")
    print(f"  RESUMEN EKF + KoopmanMPC — PLANTA REAL")
    print(f"  {'='*55}")
    print(f"  RMSE seguimiento h₂ (SS):    {rmse_ctrl:.4f} cm")
    print(f"  RMSE estimación  h₁ (SS):    {rmse_h1:.4f} cm  ← EKF vs medición")
    print(f"  RMSE estimación  h₃ (SS):    {rmse_h3:.4f} cm  ← EKF vs medición")
    print(f"  RMSE estimación  h₂ (SS):    {rmse_h2_obs:.4f} cm  ← EKF vs verdad terreno")
    print(f"  Bias  estimación h₂ (SS):    {bias_h2_obs:.4f} cm")
    print(f"  MAE   estimación h₂ (SS):    {mae_h2_obs:.4f} cm")
    print(f"  {'─'*55}")
    print(f"  Tiempo EKF/paso (media):     {avg_ekf:.2f} ms")
    print(f"  Tiempo MPC/paso (media):     {avg_mpc:.1f} ms")
    print(f"  Tiempo TOTAL/paso (media):   {avg_tot:.1f} ms")
    print(f"  Tiempo TOTAL/paso (p99):     {p99_tot:.1f} ms")
    print(f"  Outliers rechazados:         {res['outliers']}")
    print(f"  {'='*55}")
    print(f"\n  TABLA PARA EL ARTÍCULO (copiar y pegar):")
    print(f"  {'Métrica':<35} {'EKF+Koopman':>12}")
    print(f"  {'-'*48}")
    print(f"  {'RMSE ctrl h2 SS [cm]':<35} {rmse_ctrl:>12.3f}")
    print(f"  {'RMSE est. h1 SS [cm]':<35} {rmse_h1:>12.3f}")
    print(f"  {'RMSE est. h3 SS [cm]':<35} {rmse_h3:>12.3f}")
    print(f"  {'RMSE obs. h2 SS [cm]':<35} {rmse_h2_obs:>12.3f}")
    print(f"  {'Tiempo EKF [ms]':<35} {avg_ekf:>12.2f}")
    print(f"  {'Tiempo MPC [ms]':<35} {avg_mpc:>12.1f}")
    print(f"  {'Tiempo total [ms]':<35} {avg_tot:>12.1f}")
    print(f"  {'Tiempo total p99 [ms]':<35} {p99_tot:>12.1f}")

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("="*65)
    print("  EKF + KoopmanMPC — PLANTA FÍSICA REAL")
    print(f"  Puerto: {SERIAL_PORT} | Ts={Ts}s | T_sim={T_sim}s | {Nsim} pasos")
    print("="*65)

    # 1. Puntos estacionarios
    print("\nCalculando puntos de operación nominales...")
    SS_NOM = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS_NOM[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)} | u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → ✗ No convergió")

    if not SS_NOM:
        print("  ERROR: Sin puntos estacionarios. Revisa P_NOM.")
        sys.exit(1)

    # 2. EDMD (mismo que Koopman CL — para comparación justa)
    A_d, B_d, nz, nu = train_edmd(verbose=True)

    # 3. KoopmanMPC (idéntico al de koopman_real_plant.py)
    mpc_args = build_mpc(A_d, B_d, nz, nu)
    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz2, nu2 = mpc_args

    # 4. Conectar al ESP32
    esp = ESP32Interface(port=SERIAL_PORT, baud=BAUD_RATE)
    try:
        esp.connect()

        print("\n  Verificación de sensores (5 lecturas):")
        for i in range(5):
            h1, h2_real, h3 = esp.get_measurement()
            print(f"    [{i+1}] h1={h1*100:.2f}cm | h2_real={h2_real*100:.2f}cm | h3={h3*100:.2f}cm")
            time.sleep(0.5)

        print(f"\n  EKF configurado con:")
        print(f"    Q_h2 = {np.sqrt(EKF_Q[1,1])*1000:.1f} mm (ruido proceso h2)")
        print(f"    R    = {np.sqrt(EKF_R[0,0])*1000:.1f} mm (ruido medición)")
        print(f"    P0_h2= {np.sqrt(EKF_P0[1,1])*1000:.1f} mm (incertidumbre inicial h2)")

        input("\n  ¿Todo OK? Presiona ENTER para iniciar (Ctrl+C para cancelar)...")

        # 5. Experimento
        res = run_ekf_experiment(
            esp, A_d, B_d, solver, N_mpc,
            umin, umax, lbw, ubw, lbg, ubg, nz2, nu2, SS_NOM
        )

        # 6. Resultados
        print_summary(res)
        plot_results(res)

    except KeyboardInterrupt:
        print("\n\n  Interrumpido por el usuario.")
    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback; traceback.print_exc()
    finally:
        esp.close()

if __name__ == '__main__':
    main()

  EKF + KoopmanMPC — PLANTA FÍSICA REAL
  Puerto: COM6 | Ts=2.0s | T_sim=800s | 400 pasos

Calculando puntos de operación nominales...
  h2=0.10m → x_ss=[0.1139 0.1    0.0818] | u_ss=[0.3283 0.0012] ✓
  h2=0.15m → x_ss=[0.1675 0.15   0.1269] | u_ss=[0.4029 0.0012] ✓
  h2=0.20m → x_ss=[0.2207 0.2    0.1727] | u_ss=[0.4659 0.0012] ✓

Entrenando EDMD (parámetros nominales)...
  8220 muestras de entrenamiento
  RMSE one-step: 9.74e-05
  |λ| ∈ [0.03444, 0.99990] ✓

Construyendo KoopmanMPC...
  KoopmanMPC ✓

Conectando a COM6 @ 115200 baud...
  ✓ Conectado a COM6

  Verificación de sensores (5 lecturas):
    [1] h1=1.91cm | h2_real=1.59cm | h3=1.08cm
    [2] h1=1.91cm | h2_real=1.57cm | h3=1.08cm
    [3] h1=1.91cm | h2_real=1.59cm | h3=1.08cm
    [4] h1=1.91cm | h2_real=1.59cm | h3=1.08cm
    [5] h1=1.91cm | h2_real=1.59cm | h3=1.08cm

  EKF configurado con:
    Q_h2 = 2.0 mm (ruido proceso h2)
    R    = 1.5 mm (ruido medición)
    P0_h2= 20.0 mm (incertidumbre inicial h2)

Esperando primer